In [17]:
# =============================================================================
# Extract Course Domain — VI, EN, FR, DE, ZH, ES, PT
# Dataset: Multilingual-NLP/M-ABSA
# =============================================================================
# Chạy: python extract_course_multilingual.py
# Hoặc copy từng section vào Jupyter notebook
# =============================================================================

## 1. Cài đặt & Import

In [18]:
# datasets và pandas đã có sẵn, chỉ cài thêm langid (~1MB, nhanh)
# !pip install -q langid

import ast
import os
import pandas as pd
import langid
from datasets import load_dataset

# Giới hạn langid trong 21 ngôn ngữ của dataset → chính xác hơn
DATASET_LANGS = ["ar", "da", "de", "en", "es", "fr", "hi", "hr",
                 "id", "ja", "ko", "nl", "pt", "ru", "sk", "sv",
                 "sw", "th", "tr", "vi", "zh"]
langid.set_languages(DATASET_LANGS)

print("Loading M-ABSA dataset...")
ds = load_dataset("Multilingual-NLP/M-ABSA")
print("Done!", {k: len(v) for k, v in ds.items()})

Loading M-ABSA dataset...
Done! {'train': 184716, 'validation': 45906, 'test': 79674}


## 2. Gộp tất cả splits

In [19]:
all_dfs = []
for split_name in ds.keys():
    df_tmp = ds[split_name].to_pandas()
    df_tmp["split"] = split_name
    all_dfs.append(df_tmp)

df_all = pd.concat(all_dfs, ignore_index=True)
print(f"Total rows: {len(df_all):,}")

Total rows: 310,296


## 3. Parse sentence & labels

In [20]:
def parse_row(raw_text):
    """Tách sentence và labels từ: sentence####[[aspect, category, sentiment]]"""
    raw_text = str(raw_text)
    if "####" in raw_text:
        sentence, label_str = raw_text.split("####", 1)
        try:
            labels = ast.literal_eval(label_str.strip())
        except Exception:
            labels = []
    else:
        sentence = raw_text
        labels = []
    return sentence.strip(), labels

df_all[["sentence", "labels"]] = df_all["text"].apply(
    lambda x: pd.Series(parse_row(x))
)
print(f"Parsed xong! Shape: {df_all.shape}")

Parsed xong! Shape: (310296, 4)


## 4. Khám phá tất cả aspect categories trong dataset

In [21]:
def extract_categories(text):
    if "####" not in str(text):
        return []
    try:
        labels = ast.literal_eval(text.split("####", 1)[1].strip())
        return [triplet[1] for triplet in labels if len(triplet) >= 2]
    except:
        return []

df_all["categories"] = df_all["text"].apply(extract_categories)

all_cats = set()
df_all["categories"].apply(lambda cats: all_cats.update(cats))
print(f"Total unique categories: {len(all_cats)}")
print("\nAll categories:")
for c in sorted(all_cats):
    print(f"  - {c}")

Total unique categories: 283

All categories:
  - After-sales Service#Exchange/Warranty/Return
  - Appearance Design#Aesthetics General
  - Appearance Design#Color
  - Appearance Design#Exterior Design Material
  - Appearance Design#Fuselage Size
  - Appearance Design#Grip Feeling
  - Appearance Design#Thickness
  - Appearance Design#Weight
  - Appearance Design#Workmanship and Texture
  - Audio/Sound#Tone quality
  - Audio/Sound#Volume and Speaker
  - BATTERY#DESIGN_FEATURES
  - BATTERY#GENERAL
  - BATTERY#OPERATION_PERFORMANCE
  - BATTERY#QUALITY
  - Battery/Long evity#Battery Life
  - Battery/Longevity#Battery Capacity
  - Battery/Longevity#Battery Life
  - Battery/Longevity#Charging Method
  - Battery/Longevity#Charging Speed
  - Battery/Longevity#General
  - Battery/Longevity#Power Consumption Speed
  - Battery/Longevity#Standby Time
  - Branding/Marketing#Promotional Giveaways
  - Buyer Attitude#Loyalty
  - Buyer Attitude#Recommendable
  - Buyer Attitude#Repurchase and Churn Tend

## 5. Lọc domain Coursera theo aspect category

In [22]:
COURSERA_CATEGORIES_LOWER = {
    "assignments comprehensiveness", "assignments quality", "assignments quantity",
    "assignments relatability",      "assignments workload",
    "course comprehensiveness",      "course general",      "course quality",
    "course relatability",           "course value",        "course workload",
    "faculty comprehensiveness",     "faculty general",     "faculty relatability",
    "faculty response",              "faculty value",
    "grades general",
    "material comprehensiveness",    "material quality",    "material quantity",
    "material relatability",         "material workload",
    "presentation comprehensiveness","presentation quality","presentation quantity",
    "presentation relatability",     "presentation workload",
    "course_general_feedback",
    "instructor",
    "mathematical_related_concept",
    "teaching_setup",
    "facilities general",   "facilities quality",  "facilities cleanliness",
    "facilities comfort",   "facilities design_features",
    "facilities miscellaneous", "facilities prices",
    "polarity positive",    "polarity negative",   "polarity neutral",
    "proyecto final",
}

def is_coursera(labels):
    if not labels:
        return False
    for triplet in labels:
        if len(triplet) >= 2 and str(triplet[1]).lower() in COURSERA_CATEGORIES_LOWER:
            return True
    return False

df_all["is_coursera"] = df_all["labels"].apply(is_coursera)
df_course = df_all[df_all["is_coursera"]].copy()
print(f"Rows sau khi lọc domain coursera: {len(df_course):,}")
print(df_course["split"].value_counts().to_string())

Rows sau khi lọc domain coursera: 68,346
split
train         36927
test          22004
validation     9415


## 6. Detect ngôn ngữ

Bước 1 — Unicode script (chính xác 100%, không cần model):
  Hiragana/Katakana → ja | CJK only → zh | Hangul → ko
  Arabic → ar | Thai → th | Devanagari → hi | Cyrillic → ru

Bước 2 — langid cho các ngôn ngữ Latin (VI/EN/FR/DE/ES/PT/...)

In [23]:
def has_script(text, start, end):
    return any(start <= ord(c) <= end for c in text)

def detect_by_script(text):
    t = str(text)
    if has_script(t, 0xAC00, 0xD7A3) or has_script(t, 0x1100, 0x11FF):
        return "ko"   # Hangul
    if has_script(t, 0x3040, 0x309F) or has_script(t, 0x30A0, 0x30FF):
        return "ja"   # Hiragana / Katakana
    if has_script(t, 0x4E00, 0x9FFF) or has_script(t, 0x3400, 0x4DBF):
        return "zh"   # CJK Unified Ideographs
    if has_script(t, 0x0600, 0x06FF):
        return "ar"   # Arabic
    if has_script(t, 0x0E00, 0x0E7F):
        return "th"   # Thai
    if has_script(t, 0x0900, 0x097F):
        return "hi"   # Devanagari
    if has_script(t, 0x0400, 0x04FF):
        return "ru"   # Cyrillic
    return None       # Latin → dùng langid

def detect_language(text):
    script = detect_by_script(text)
    if script is not None:
        return script
    lang, _ = langid.classify(str(text))
    return lang

# Kiểm tra nhanh
tests = [
    ("Python の非常に詳細な入門学習。",       "ja"),
    ("这门课程非常好。",                       "zh"),
    ("Nó cung cấp lời khuyên hữu ích.",      "vi"),
    ("Excellent course.",                     "en"),
    ("Ansonsten war es super.",               "de"),
    ("Tak Coursera og Michigan University.", "da"),
    ("Un cours absolument fantastique.",      "fr"),
    ("Un curso absolutamente fantástico.",    "es"),
    ("Um curso absolutamente fantástico.",    "pt"),
]
print("Test detect_language:")
for text, expected in tests:
    result = detect_language(text)
    status = "✓" if result == expected else "✗"
    print(f"  {status} [{result:3s}] (expected {expected:3s}) {text}")

Test detect_language:
  ✓ [ja ] (expected ja ) Python の非常に詳細な入門学習。
  ✓ [zh ] (expected zh ) 这门课程非常好。
  ✓ [vi ] (expected vi ) Nó cung cấp lời khuyên hữu ích.
  ✗ [de ] (expected en ) Excellent course.
  ✓ [de ] (expected de ) Ansonsten war es super.
  ✗ [id ] (expected da ) Tak Coursera og Michigan University.
  ✓ [fr ] (expected fr ) Un cours absolument fantastique.
  ✓ [es ] (expected es ) Un curso absolutamente fantástico.
  ✗ [es ] (expected pt ) Um curso absolutamente fantástico.


In [24]:
print(f"Detecting language cho {len(df_course):,} rows...")
df_course = df_course.copy()
df_course["language"] = df_course["sentence"].apply(detect_language)
print("Done!")

print("\nPhân phối ngôn ngữ (tất cả):")
print(df_course["language"].value_counts().to_string())

Detecting language cho 68,346 rows...
Done!

Phân phối ngôn ngữ (tất cả):
language
en    3485
es    3367
de    3353
zh    3286
id    3282
ja    3262
ru    3259
hi    3258
ko    3258
vi    3258
ar    3257
th    3257
sk    3256
fr    3249
sv    3237
da    3231
hr    3228
tr    3188
nl    3185
pt    3168
sw    3022


## 7. Lọc ngôn ngữ: VI, EN, FR, DE, ZH, ES, PT

In [25]:
TARGET_LANGS = ["vi", "en", "fr", "de", "zh", "es", "pt"]

df_filtered = df_course[df_course["language"].isin(TARGET_LANGS)].copy()
print(f"Rows sau khi lọc ngôn ngữ: {len(df_filtered):,}")

Rows sau khi lọc ngôn ngữ: 23,166


## 8. Thống kê & kiểm tra chất lượng

In [26]:
lang_names = {
    "vi": "Tiếng Việt",        "en": "Tiếng Anh",
    "fr": "Tiếng Pháp",        "de": "Tiếng Đức",
    "zh": "Tiếng Trung",       "es": "Tiếng Tây Ban Nha",
    "pt": "Tiếng Bồ Đào Nha",
}

print("=" * 55)
print("KẾT QUẢ SAU KHI LỌC")
print("=" * 55)
print(f"Tổng số câu: {len(df_filtered):,}")

print("\nPhân phối theo ngôn ngữ:")
for lang, cnt in df_filtered["language"].value_counts().items():
    print(f"  {lang.upper():5s} ({lang_names.get(lang, lang)}): {cnt:,} câu")

print("\nPhân phối theo split:")
print(df_filtered["split"].value_counts().to_string())

print("\nNgôn ngữ × Split:")
print(df_filtered.groupby(["language", "split"]).size().unstack(fill_value=0).to_string())

KẾT QUẢ SAU KHI LỌC
Tổng số câu: 23,166

Phân phối theo ngôn ngữ:
  EN    (Tiếng Anh): 3,485 câu
  ES    (Tiếng Tây Ban Nha): 3,367 câu
  DE    (Tiếng Đức): 3,353 câu
  ZH    (Tiếng Trung): 3,286 câu
  VI    (Tiếng Việt): 3,258 câu
  FR    (Tiếng Pháp): 3,249 câu
  PT    (Tiếng Bồ Đào Nha): 3,168 câu

Phân phối theo split:
split
train         12528
test           7457
validation     3181

Ngôn ngữ × Split:
split     test  train  validation
language                         
de        1068   1810         475
en        1124   1893         468
es        1089   1822         456
fr        1055   1750         444
pt        1021   1706         441
vi        1048   1763         447
zh        1052   1784         450


In [27]:
# Xem 3 ví dụ cho mỗi ngôn ngữ
for lang in TARGET_LANGS:
    subset = df_filtered[df_filtered["language"] == lang]
    if len(subset) == 0:
        print(f"[{lang.upper()}] Không có câu nào.")
        continue
    print(f"\n{'='*55}")
    print(f"[{lang.upper()}] {lang_names.get(lang, lang)} — {len(subset):,} câu")
    print(f"{'='*55}")
    for _, row in subset.sample(min(3, len(subset)), random_state=42).iterrows():
        print(f"  Sentence : {row['sentence'][:150]}")
        print(f"  Labels   : {row['labels']}")
        print()


[VI] Tiếng Việt — 3,258 câu
  Sentence : xin lỗi tôi bị lạc ở 32"14, chính xác thì chuyện gì đã xảy ra với cách anh ấy lấy được hàng cuối cùng của ma trận gần như là một bản sắc?
  Labels   : [['NULL', 'Instructor', 'NEU'], ['ma trận', 'Mathematical_Related_Concept', 'NEU']]

  Sentence : Thật là thử thách.
  Labels   : [['NULL', 'polarity positive', 'positive']]

  Sentence : Tuy nhiên, tôi đã học được nhiều điều mới ngay từ đầu.
  Labels   : [['NULL', 'course general', 'positive']]


[EN] Tiếng Anh — 3,485 câu
  Sentence : This course on nutrition , including organic / holistic options , gives practical , easy - to - use ideas on healthy and enjoyable eating options not 
  Labels   : [['course', 'course quality', 'positive']]

  Sentence : this guy is awesome the proffs at the university of toronto dont know shit abt math...
  Labels   : [['NULL', 'Instructor', 'POS'], ['proffs', 'Instructor', 'NEG']]

  Sentence : I love it .
  Labels   : [['NULL', 'course general', 'positive']]




## 9. Normalize sentiment labels (POS/NEG/NEU → positive/negative/neutral)

In [28]:
SENTIMENT_MAP = {
    "pos": "positive",
    "neg": "negative",
    "neu": "neutral",
}

def normalize_labels(labels):
    result = []
    for triplet in labels:
        if len(triplet) == 3:
            aspect, category, sentiment = triplet
            sentiment_norm = SENTIMENT_MAP.get(str(sentiment).lower(), str(sentiment).lower())
            result.append([aspect, category, sentiment_norm])
        else:
            result.append(triplet)
    return result

df_filtered["labels"] = df_filtered["labels"].apply(normalize_labels)

# Kiểm tra các giá trị sentiment sau normalize
all_sentiments = set()
df_filtered["labels"].apply(lambda lbls: [all_sentiments.add(t[2]) for t in lbls if len(t) == 3])
print("Unique sentiment values sau normalize:", sorted(all_sentiments))

# Xem ví dụ ZH và ES sau normalize
for lang in ["zh", "es"]:
    sample = df_filtered[df_filtered["language"] == lang].head(2)
    print(f"\n[{lang.upper()}] sample:")
    for _, row in sample.iterrows():
        print(f"  {row['labels']}")

Unique sentiment values sau normalize: ['conflict', 'negative', 'neutral', 'positive']

[ZH] sample:
  [['教授', 'faculty general', 'positive']]
  [['内容', 'course general', 'positive']]

[ES] sample:
  [['NULL', 'course general', 'positive']]
  [['Mr. George Siedel', 'faculty general', 'positive']]


## 10. Map sang 15 aspect labels & tạo output format

LECTURER:   0=Teaching_Skill  1=Knowledge  2=Experience  3=Behavior  4=Support
COURSE:     5=Curriculum      6=Materials  7=Workload    8=Assignments
ASSESSMENT: 9=Grading        10=Exams
FACILITIES: 11=Classroom     12=Platforms
OTHERS:     13=General       14=Recommendation

In [29]:
ASPECT_LABELS = [
    "Teaching_Skill",  # 0  - LECTURER
    "Knowledge",       # 1
    "Experience",      # 2
    "Behavior",        # 3
    "Support",         # 4
    "Curriculum",      # 5  - COURSE
    "Materials",       # 6
    "Workload",        # 7
    "Assignments",     # 8
    "Grading",         # 9  - ASSESSMENT
    "Exams",           # 10
    "Classroom",       # 11 - FACILITIES
    "Platforms",       # 12
    "General",         # 13 - OTHERS
    "Recommendation",  # 14
]

CATEGORY_TO_ASPECT = {
    # ── LECTURER: Teaching_Skill (0) ──────────────────────────
    "faculty general":                0,
    "faculty comprehensiveness":      0,
    "faculty relatability":           0,
    "faculty value":                  0,
    "teaching_setup":                 0,
    "presentation quality":           0,
    "presentation quantity":          0,
    "presentation comprehensiveness": 0,
    "presentation relatability":      0,
    "presentation workload":          0,
    # ── LECTURER: Knowledge (1) ───────────────────────────────
    "mathematical_related_concept":   1,
    # ── LECTURER: Experience (2) ──────────────────────────────
    "instructor":                     2,
    # ── LECTURER: Support (4) ─────────────────────────────────
    "faculty response":               4,
    # ── COURSE: Curriculum (5) ────────────────────────────────
    "course general":                 5,
    "course quality":                 5,
    "course comprehensiveness":       5,
    "course relatability":            5,
    "course value":                   5,
    "course_general_feedback":        5,
    # ── COURSE: Materials (6) ─────────────────────────────────
    "material quality":               6,
    "material quantity":              6,
    "material comprehensiveness":     6,
    "material relatability":          6,
    "material workload":              6,
    # ── COURSE: Workload (7) ──────────────────────────────────
    "course workload":                7,
    # ── COURSE: Assignments (8) ───────────────────────────────
    "assignments comprehensiveness":  8,
    "assignments quality":            8,
    "assignments quantity":           8,
    "assignments relatability":       8,
    "assignments workload":           8,
    "proyecto final":                 8,
    # ── ASSESSMENT: Grading (9) ───────────────────────────────
    "grades general":                 9,
    "polarity positive":              9,
    "polarity negative":              9,
    "polarity neutral":               9,
    # ── FACILITIES: Classroom (11) ────────────────────────────
    "facilities general":             11,
    "facilities quality":             11,
    "facilities cleanliness":         11,
    "facilities comfort":             11,
    "facilities design_features":     11,
    "facilities miscellaneous":       11,
    "facilities prices":              11,
    # ── OTHERS: General (13) ──────────────────────────────────
    "null":                           13,
    "other":                          13,
}

# Keyword heuristic → Recommendation (14)
RECOMMEND_KEYWORDS = [
    "recommend", "suggest", "worth", "should take", "must take",
    "giới thiệu", "nên học", "đáng học",
    "recommande", "conseille",
    "empfehle", "empfehlen",
    "recomiend", "recomiendo",
    "recomend", "recomendo",
    "推荐",
]
RECOMMEND_ELIGIBLE = {
    "course general", "course quality", "course_general_feedback", "null", "other"
}

def map_category_to_aspect(category, sentence=""):
    """Trả về (aspect_name, aspect_label_index) hoặc (None, None)."""
    cat_lower  = str(category).lower().strip()
    sent_lower = str(sentence).lower()

    # Heuristic Recommendation
    if cat_lower in RECOMMEND_ELIGIBLE:
        if any(kw in sent_lower for kw in RECOMMEND_KEYWORDS):
            return ASPECT_LABELS[14], 14

    idx = CATEGORY_TO_ASPECT.get(cat_lower)
    if idx is None:
        return None, None
    return ASPECT_LABELS[idx], idx

# ── Test mapping ──────────────────────────────────────────────
test_cases = [
    ("faculty general",              "",                                   0),
    ("Mathematical_Related_Concept", "",                                   1),
    ("Instructor",                   "",                                   2),
    ("faculty response",             "",                                   4),
    ("course general",               "",                                   5),
    ("material quality",             "",                                   6),
    ("course workload",              "",                                   7),
    ("assignments quality",          "",                                   8),
    ("grades general",               "",                                   9),
    ("facilities general",           "",                                  11),
    ("NULL",                         "",                                  13),
    ("Teaching_Setup",               "",                                   0),
    ("presentation quality",         "",                                   0),
    ("course general",               "I would recommend this course",     14),
    ("course general",               "Tôi muốn giới thiệu khóa học này",  14),
]
print("Test mapping:")
all_ok = True
for cat, sent, expected_idx in test_cases:
    name, idx = map_category_to_aspect(cat, sent)
    ok = (idx == expected_idx)
    if not ok:
        all_ok = False
    status = "✓" if ok else f"✗ (expected {expected_idx})"
    print(f"  {status} [{str(idx):>2}] {str(name):<20} ← '{cat}'")
print("\nAll tests passed!" if all_ok else "\nCó lỗi mapping, kiểm tra lại!")

Test mapping:
  ✓ [ 0] Teaching_Skill       ← 'faculty general'
  ✓ [ 1] Knowledge            ← 'Mathematical_Related_Concept'
  ✓ [ 2] Experience           ← 'Instructor'
  ✓ [ 4] Support              ← 'faculty response'
  ✓ [ 5] Curriculum           ← 'course general'
  ✓ [ 6] Materials            ← 'material quality'
  ✓ [ 7] Workload             ← 'course workload'
  ✓ [ 8] Assignments          ← 'assignments quality'
  ✓ [ 9] Grading              ← 'grades general'
  ✓ [11] Classroom            ← 'facilities general'
  ✓ [13] General              ← 'NULL'
  ✓ [ 0] Teaching_Skill       ← 'Teaching_Setup'
  ✓ [ 0] Teaching_Skill       ← 'presentation quality'
  ✓ [14] Recommendation       ← 'course general'
  ✓ [14] Recommendation       ← 'course general'

All tests passed!


In [30]:
# TRANSFORM: mỗi triplet → 1 row
# Columns: text | aspect | entity | attribute | aspect_label | sentiment | language | split

rows = []
for _, row in df_filtered.iterrows():
    sentence = row["sentence"]
    language = row["language"]
    split    = row["split"]

    for triplet in row["labels"]:
        if len(triplet) != 3:
            continue
        entity, attribute, sentiment = triplet

        aspect_name, aspect_idx = map_category_to_aspect(attribute, sentence)
        if aspect_name is None:
            continue  # bỏ qua category không thuộc giáo dục

        rows.append({
            "text":         sentence,
            "aspect":       aspect_name,
            "entity":       entity,
            "attribute":    attribute,
            "aspect_label": aspect_idx,
            "sentiment":    sentiment,
            "language":     language,
            "split":        split,
        })

df_output = pd.DataFrame(rows)
print(f"Total rows (1 triplet = 1 row): {len(df_output):,}")

print("\nPhân phối aspect:")
aspect_dist = (
    df_output.groupby(["aspect_label", "aspect"])
    .size().reset_index(name="count")
    .sort_values("aspect_label")
)
for _, r in aspect_dist.iterrows():
    print(f"  [{int(r.aspect_label):2d}] {r.aspect:<20s}: {r['count']:,}")

print("\nPhân phối sentiment:")
print(df_output["sentiment"].value_counts().to_string())

print("\nPhân phối ngôn ngữ:")
print(df_output["language"].value_counts().to_string())

print("\nSample output:")
print(df_output.head(10).to_string())

Total rows (1 triplet = 1 row): 33,621

Phân phối aspect:
  [ 0] Teaching_Skill      : 5,298
  [ 1] Knowledge           : 5,369
  [ 2] Experience          : 4,792
  [ 4] Support             : 309
  [ 5] Curriculum          : 12,112
  [ 6] Materials           : 1,239
  [ 7] Workload            : 119
  [ 8] Assignments         : 1,347
  [ 9] Grading             : 258
  [11] Classroom           : 985
  [13] General             : 1,120
  [14] Recommendation      : 673

Phân phối sentiment:
sentiment
positive    21906
neutral      6294
negative     5407
conflict       14

Phân phối ngôn ngữ:
language
en    4991
es    4883
de    4854
zh    4782
vi    4752
fr    4712
pt    4647

Sample output:
                                                        text          aspect             entity             attribute  aspect_label sentiment language  split
0                          Intro session var meget gentagne.  Teaching_Skill               NULL  presentation quality             0  negative     

## 11. Lưu kết quả

In [31]:
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

save_cols = ["text", "aspect", "entity", "attribute", "aspect_label", "sentiment", "language", "split"]

# 1. Toàn bộ
path = os.path.join(output_dir, "course_vi_en_fr_de_zh_es_pt_all.csv")
df_output[save_cols].to_csv(path, index=False, encoding="utf-8-sig")
print(f"[ALL]        {path}  ({len(df_output):,} rows)")

# 2. Theo ngôn ngữ
for lang in TARGET_LANGS:
    df_lang = df_output[df_output["language"] == lang]
    if len(df_lang) == 0:
        continue
    path = os.path.join(output_dir, f"course_{lang}.csv")
    df_lang[save_cols].to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[{lang.upper():5s}]       {path}  ({len(df_lang):,} rows)")

# 3. Theo split
for sp in ["train", "validation", "test"]:
    df_sp = df_output[df_output["split"] == sp]
    if len(df_sp) == 0:
        continue
    path = os.path.join(output_dir, f"course_multilingual_{sp}.csv")
    df_sp[save_cols].to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[{sp:12s}] {path}  ({len(df_sp):,} rows)")

print("\nDone! Tất cả file đã lưu vào thư mục 'output/'")

[ALL]        output/course_vi_en_fr_de_zh_es_pt_all.csv  (33,621 rows)
[VI   ]       output/course_vi.csv  (4,752 rows)
[EN   ]       output/course_en.csv  (4,991 rows)
[FR   ]       output/course_fr.csv  (4,712 rows)
[DE   ]       output/course_de.csv  (4,854 rows)
[ZH   ]       output/course_zh.csv  (4,782 rows)
[ES   ]       output/course_es.csv  (4,883 rows)
[PT   ]       output/course_pt.csv  (4,647 rows)
[train       ] output/course_multilingual_train.csv  (18,010 rows)
[validation  ] output/course_multilingual_validation.csv  (4,709 rows)
[test        ] output/course_multilingual_test.csv  (10,902 rows)

Done! Tất cả file đã lưu vào thư mục 'output/'
